# 1. Environment Setup & Data Loading

## 1.1 Project configuration and global settings

In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import math
import numpy as np

from sklearn.model_selection import train_test_split

In [2]:
def setup_data():
    org_path = "C:/Users/Owner/Desktop/지윤/3.기타/프로젝트/purchase-conversion-prediction/"
    raw_path = f"{org_path}data/raw/"
    prc_path = f"{org_path}data/processed/"
    seed = 42

    print("변수 세팅 완료")
    return org_path, raw_path, prc_path, seed

# 사용 예시 (반환값 9개)
org_path, raw_path, prc_path, seed = setup_data()

변수 세팅 완료


In [3]:
pd.set_option("display.max_columns", None)

## 1.2 Loading raw source files (CSV)

In [4]:
def load_multiple_csv(base_path: str, filenames: list[str]) -> dict[str, pd.DataFrame]:
    """
    여러 CSV 파일을 한 번에 불러와 DataFrame 딕셔너리로 반환합니다.
    
    Parameters
    ----------
    base_path : str
        기본 경로 (예: "data/raw/")
    filenames : list[str]
        확장자를 제외한 파일명 리스트 (예: ["orders", "products", "users"])

    Returns
    -------
    dict[str, pd.DataFrame]
        각 파일명을 key, DataFrame을 value로 하는 딕셔너리
    """
    base = Path(base_path)
    dataframes = {}

    for name in filenames:
        file_path = base / f"{name}.csv"
        if not file_path.exists():
            print(f"파일 없음: {file_path}")
            continue
        df = pd.read_csv(file_path)
        dataframes[name] = df
        print(f"{name}.csv 로드 완료. shape={df.shape}")

    return dataframes

In [5]:
file_list = [
    "distribution_centers", "events", "inventory_items",
    "order_items", "orders", "products", "users"
]

# CSV 로드
dfs = load_multiple_csv(raw_path, file_list)

distribution_centers.csv 로드 완료. shape=(10, 4)
events.csv 로드 완료. shape=(2431963, 13)
inventory_items.csv 로드 완료. shape=(490705, 12)
order_items.csv 로드 완료. shape=(181759, 11)
orders.csv 로드 완료. shape=(125226, 9)
products.csv 로드 완료. shape=(29120, 9)
users.csv 로드 완료. shape=(100000, 15)


In [6]:
# 변수 자동 생성
for name, df in dfs.items():
    globals()[name] = df
    print(f"변수 생성 완료: {name} (shape={df.shape})")

변수 생성 완료: distribution_centers (shape=(10, 4))
변수 생성 완료: events (shape=(2431963, 13))
변수 생성 완료: inventory_items (shape=(490705, 12))
변수 생성 완료: order_items (shape=(181759, 11))
변수 생성 완료: orders (shape=(125226, 9))
변수 생성 완료: products (shape=(29120, 9))
변수 생성 완료: users (shape=(100000, 15))


# 2. Raw Data Preprocessing

세션 단위로 예측이 필요하므로 세션 단위로 집계 필요
1) 파생 변수 생성 및 가공
1) 이벤트 개수 기반 Feature
2) 시간/행동 패턴 Feature
3) 결과 이벤트 Feature (Label 생성용)


## 2.1 Event-level preprocessing

In [7]:
def preprocess_events(events: pd.DataFrame):
    events = events.copy()

    # 1) 기본 dtype 정리
    events = events.convert_dtypes()

    force_dtypes = {
        'id': 'string',
        'user_id': 'string',
        'sequence_number': 'string',
    }
    events = events.astype({k: v for k, v in force_dtypes.items() if k in events.columns})

    # 2) 시간 컬럼 정규화 + 파생
    events['created_at'] = pd.to_datetime(events['created_at'], errors='coerce')

    # 모델링/EDA 둘 다 쓰기 좋은 형태: "수치형"으로 보존
    events['year']   = events['created_at'].dt.year.astype('Int64')
    events['month']  = events['created_at'].dt.month.astype('Int64')
    events['day']    = events['created_at'].dt.day.astype('Int64')
    events['hour']   = events['created_at'].dt.hour.astype('Int64')
    events['minute'] = events['created_at'].dt.minute.astype('Int64')
    events['second'] = events['created_at'].dt.second.astype('Int64')

    derived_time_cols = ['year', 'month', 'day', 'hour', 'minute', 'second']

    # 3) 사용자 ID 존재 여부 (boolean 유지)
    events['is_user_id_present'] = events['user_id'].notna()

    # 4) URI 파생 변수
    def extract_first_two_segments(uri):
        uri = str(uri)
        segs = uri.split('/')
        return f"/{segs[1]}" if len(segs) >= 2 and segs[1] != '' else uri

    events['uri_splt'] = events['uri'].apply(extract_first_two_segments)

    # 5) 유저별 생성 세션 수
    events['unique_session_count'] = (
        events.groupby('user_id')['session_id']
              .transform('nunique')
    )

    # 6) 한 세션별 활동 수
    events['activity_count'] = (
        events.groupby('session_id')['session_id']
              .transform('size')
    )

    # 7) 컬럼 그룹 정리
    key_cols = ['id', 'user_id', 'session_id', 'ip_address']
    key_cols = [c for c in key_cols if c in events.columns]

    date_cols = ['created_at']

    # 범주형 후보: 문자열 + 카테고리 + boolean
    cat_cols = events.select_dtypes(include=['string', 'category', 'boolean']).columns.tolist()

    # key, date는 제외
    cat_cols = [c for c in cat_cols if c not in key_cols + date_cols]

    # 필요하면 derived_time_cols를 "범주형처럼" 쓰고 싶을 때만 아래처럼 캐스팅해서 사용
    # (기본 전처리 단계에서는 수치형 유지가 더 유연함)
    # for c in derived_time_cols:
    #     events[c] = events[c].astype('Int64').astype('string')
    #     if c not in cat_cols:
    #         cat_cols.append(c)

    print("범주형 컬럼:", cat_cols)
    print("키 컬럼:", key_cols)
    print("시간 컬럼:", date_cols, derived_time_cols)

    return events, key_cols, date_cols, cat_cols

In [8]:
events, key_cols, date_cols, cat_cols = preprocess_events(events)

범주형 컬럼: ['sequence_number', 'city', 'state', 'postal_code', 'browser', 'traffic_source', 'uri', 'event_type', 'is_user_id_present']
키 컬럼: ['id', 'user_id', 'session_id', 'ip_address']
시간 컬럼: ['created_at'] ['year', 'month', 'day', 'hour', 'minute', 'second']


## 2.2 Session-level Dataset Construction

In [9]:
import pandas as pd

def build_session_dataset(
    events: pd.DataFrame,
    only_identified_users: bool = True,
    only_identified_created_at: bool = True,
) -> pd.DataFrame:
    """
    이벤트 단위(events) 데이터를 세션 단위로 집계한 데이터셋 생성.

    제거된 파생 변수:
    - session_cart_events
    - session_unique_uris
    - session_unique_uri_splt
    - main_uri_splt
    - session_duration_min
    - cart_to_view_ratio

    추가로 생성하지 않는 next session 관련 정보:
    - next_session_start_time
    - label_repurchase
    - next_session_gap_min
    - next_session_gap_hour
    """

    df = events.copy()

    # 0) user_id, created_at이 있는 세션만 사용할지 여부
    if only_identified_users:
        df = df[df["user_id"].notna()].copy()

    if only_identified_created_at:
        df = df[df["created_at"].notna()].copy()

    # created_at 보정
    df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")

    # 파싱 실패한 이벤트 제거 (NaT)
    df = df[df["created_at"].notna()].copy()

    # 1) 세션 단위 집계
    session_df = (
        df.groupby(["user_id", "session_id"])
        .agg(
            # 시간
            session_start_time=("created_at", "min"),
            session_end_time=("created_at", "max"),
            # 행동량
            session_event_count=("event_type", "count"),
            session_product_views=("event_type", lambda x: (x == "product").sum()),
            session_cancel_events=("event_type", lambda x: (x == "cancel").sum()),
            event_type_nunique=("event_type", "nunique"),
            session_main_events=(
                "event_type",
                lambda x: x.value_counts().idxmax() if x.notna().any() else pd.NA,
            ),
            # 환경 정보
            session_browser=("browser", "first"),
            session_traffic_source=("traffic_source", "first"),
            session_city=("city", "first"),
            session_state=("state", "first"),
            session_postal_code=("postal_code", "first"),
        )
        .reset_index()
    )

    # 세션 단위에서도 NaT 있을 경우 방어적으로 제거
    session_df = session_df.dropna(subset=["session_start_time", "session_end_time"])

    # 2) 지속시간 (hour만 유지)
    session_df["session_duration_hour"] = (
        (session_df["session_end_time"] - session_df["session_start_time"])
        .dt.total_seconds()
        / 3600
    )

    # 3) 세션 시작/종료 시간 파생
    # --- 시작 ---
    session_df["session_start_hour"] = session_df["session_start_time"].dt.hour
    session_df["session_start_weekday"] = session_df["session_start_time"].dt.weekday
    session_df["session_start_is_weekend"] = (
        session_df["session_start_weekday"].isin([5, 6]).astype(int)
    )
    session_df["session_start_day"] = session_df["session_start_time"].dt.day
    session_df["session_start_week"] = (
        session_df["session_start_time"].dt.isocalendar().week.astype("Int64")
    )
    session_df["session_start_month"] = session_df["session_start_time"].dt.month
    session_df["session_start_quarter"] = session_df["session_start_time"].dt.quarter
    session_df["session_start_year"] = session_df["session_start_time"].dt.year

    # --- 종료 ---
    session_df["session_end_hour"] = session_df["session_end_time"].dt.hour
    session_df["session_end_weekday"] = session_df["session_end_time"].dt.weekday
    session_df["session_end_is_weekend"] = (
        session_df["session_end_weekday"].isin([5, 6]).astype(int)
    )
    session_df["session_end_day"] = session_df["session_end_time"].dt.day
    session_df["session_end_week"] = (
        session_df["session_end_time"].dt.isocalendar().week.astype("Int64")
    )
    session_df["session_end_month"] = session_df["session_end_time"].dt.month
    session_df["session_end_quarter"] = session_df["session_end_time"].dt.quarter
    session_df["session_end_year"] = session_df["session_end_time"].dt.year

    # 4) 세션 정렬 (세션 순서 계산용)
    session_df = session_df.sort_values(
        ["user_id", "session_start_time", "session_id"]
    )

    # 5) product_seq_uri
    prod = df[df["event_type"] == "product"].copy()
    prod = prod.sort_values(["user_id", "session_id", "created_at"])

    # URI 마지막 segment(상품 코드)만 추출
    def extract_last_segment(u):
        if pd.isna(u):
            return pd.NA
        s = str(u).rstrip("/").split("/")
        return s[-1] if s[-1] != "" else pd.NA

    prod["product_code"] = prod["uri"].apply(extract_last_segment)

    product_seq = (
        prod.groupby(["user_id", "session_id"])["product_code"]
        .agg(lambda x: ",".join(x.astype(str)))
        .reset_index(name="product_seq_uri")
    )

    session_df = session_df.merge(
        product_seq,
        on=["user_id", "session_id"],
        how="left",
    )

    # 6) 세션 순서
    session_df["session_order"] = (
        session_df.groupby("user_id").cumcount() + 1
    )

    # 7) 세션 타입
    session_df["session_type"] = session_df["session_order"].apply(
        lambda x: f"T{x}"
    )

    # 8) 다음 세션 존재 여부 라벨 생성 (타깃)
    session_df["next_session_start_time"] = (
        session_df.groupby("user_id")["session_start_time"].shift(-1)
    )

    session_df["label_repurchase"] = (
        session_df["next_session_start_time"].notna().astype(int)
    )

    return session_df

In [10]:
# 약 2분 30초 소요
session_df = build_session_dataset(events)
session_df

,user_id,session_id,session_start_time,session_end_time,session_event_count,session_product_views,session_cancel_events,event_type_nunique,session_main_events,session_browser,session_traffic_source,session_city,session_state,session_postal_code,session_duration_hour,session_start_hour,session_start_weekday,session_start_is_weekend,session_start_day,session_start_week,session_start_month,session_start_quarter,session_start_year,session_end_hour,session_end_weekday,session_end_is_weekend,session_end_day,session_end_week,session_end_month,session_end_quarter,session_end_year,product_seq_uri,session_order,session_type,next_session_start_time,label_repurchase
0,1,bccf01cb-6f3b-4ef7-aaff-0ea67e584334,2022-07-18 10:17:52+00:00,2022-07-20 10:32:05+00:00,10,3,0,4,product,Firefox,Adwords,Bucheon City,Gyeonggi-do,421-150,48.236944,10,0,0,18,29,7,3,2022,10,2,0,20,29,7,3,2022,"2953,2953,2953",1,T1,2022-07-18 10:52:33+00:00,1
1,1,dc670e53-0eb4-4da2-a023-8f505d74e961,2022-07-18 10:52:33+00:00,2022-07-20 11:05:38+00:00,10,3,0,4,product,Chrome,Email,Bucheon City,Gyeonggi-do,421-150,48.218056,10,0,0,18,29,7,3,2022,11,2,0,20,29,7,3,2022,"4731,4731,4731",2,T2,2022-07-18 11:18:36+00:00,1
2,1,7ed34a21-9559-4d31-a16f-d87e4c22d343,2022-07-18 11:18:36+00:00,2022-07-19 11:29:28+00:00,10,3,0,4,department,Chrome,Email,Bucheon City,Gyeonggi-do,421-150,24.181111,11,0,0,18,29,7,3,2022,11,1,0,19,29,7,3,2022,"7656,7656,7656",3,T3,NaT,0
3,100,5a795c3c-e0b6-40b3-a3ec-8dfc66157847,2023-12-09 13:20:07+00:00,2023-12-09 13:24:46+00:00,5,1,0,5,cart,Chrome,Email,Shanghai,Zhejiang,314300,0.077500,13,5,1,9,49,12,4,2023,13,5,1,9,49,12,4,2023,5742,1,T1,NaT,0
4,1000,4faaee5f-3fa9-4665-9c36-cb073c42f189,2020-04-10 06:52:02+00:00,2020-04-10 06:59:30+00:00,5,1,0,5,department,Safari,YouTube,Seoul,Seoul,138-200,0.124444,6,4,0,10,15,4,2,2020,6,4,0,10,15,4,2,2020,1936,1,T1,NaT,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
177349,99998,f900a0c8-62c5-4168-9ac8-cd0bde779778,2022-12-16 06:26:15+00:00,2022-12-16 06:32:25+00:00,5,1,0,5,product,Safari,Email,Candler,North Carolina,28715,0.102778,6,4,0,16,50,12,4,2022,6,4,0,16,50,12,4,2022,19164,1,T1,2023-01-28 06:18:17+00:00,1
177350,99998,98d1eeb2-2d59-4a94-aa42-477bd10f2f67,2023-01-28 06:18:17+00:00,2023-01-28 06:22:59+00:00,5,1,0,5,purchase,Chrome,Adwords,Candler,North Carolina,28715,0.078333,6,5,1,28,4,1,1,2023,6,5,1,28,4,1,1,2023,16938,2,T2,2023-07-22 05:04:36+00:00,1
177351,99998,841f595e-9f7b-4337-8c4c-08b82a11fc6b,2023-07-22 05:04:36+00:00,2023-07-22 05:13:13+00:00,5,1,0,5,cart,Chrome,Email,Candler,North Carolina,28715,0.143611,5,5,1,22,29,7,3,2023,5,5,1,22,29,7,3,2023,21923,3,T3,NaT,0
177352,99999,448f8d4c-7cf3-45b3-b6a7-77978e5e4c07,2023-12-22 08:51:44+00:00,2023-12-23 08:59:09+00:00,7,2,0,4,department,Chrome,YouTube,Itatiaia,Rio de Janeiro,27580-000,24.123611,8,4,0,22,51,12,4,2023,8,5,1,23,51,12,4,2023,"25959,25959",1,T1,2023-12-22 11:26:32+00:00,1


In [11]:
# 구입 상품 관련 추가 파생 변수 생성
# 세션별 유니크 상품 개수
session_df['product_seq_unique_count'] = (
    session_df['product_seq_uri']
        .fillna('')
        .apply(lambda s: len(set(s.split(','))) if s else 0)
)

# 세션별 총 상품 개수
session_df['product_seq_total_count'] = (
    session_df['product_seq_uri']
        .fillna('')
        .apply(lambda s: len(s.split(',')) if s else 0)
)

# 세션별 유니크 상품 리스트
session_df['product_seq_unique_str'] = (
    session_df['product_seq_uri']
        .fillna('')
        .apply(lambda s: ",".join(sorted(set(s.split(',')))) if s else "")
)

In [12]:
# 세션별 유니크 상품 개수는 1개. 우리가 보유한 데이터는 1개의 상품을 N개 구입함. 서로 다른 상품을 구입한 세션은 없음.
# 보통 1~4개 상품을 구입함
print(f"세션별 유니크 상품 개수 집계: {session_df['product_seq_unique_count'].value_counts()}")
print(f"세션별 총 상품 개수 집계: {session_df['product_seq_total_count'].value_counts()}")
print(f"세션별 유니크 상품 리스트 집계: {session_df['product_seq_unique_str'].value_counts()}")

세션별 유니크 상품 개수 집계: product_seq_unique_count
1    177354
Name: count, dtype: int64
세션별 총 상품 개수 집계: product_seq_total_count
1    85531
2    48594
4    24896
3    18333
Name: count, dtype: int64
세션별 유니크 상품 리스트 집계: product_seq_unique_str
18795    21
17045    19
21842    19
25209    18
27625    18
         ..
2259      1
23293     1
2514      1
17017     1
14347     1
Name: count, Length: 29033, dtype: int64


In [13]:
session_df.head()

,user_id,session_id,session_start_time,session_end_time,session_event_count,session_product_views,session_cancel_events,event_type_nunique,session_main_events,session_browser,session_traffic_source,session_city,session_state,session_postal_code,session_duration_hour,session_start_hour,session_start_weekday,session_start_is_weekend,session_start_day,session_start_week,session_start_month,session_start_quarter,session_start_year,session_end_hour,session_end_weekday,session_end_is_weekend,session_end_day,session_end_week,session_end_month,session_end_quarter,session_end_year,product_seq_uri,session_order,session_type,next_session_start_time,label_repurchase,product_seq_unique_count,product_seq_total_count,product_seq_unique_str
0,1,bccf01cb-6f3b-4ef7-aaff-0ea67e584334,2022-07-18 10:17:52+00:00,2022-07-20 10:32:05+00:00,10,3,0,4,product,Firefox,Adwords,Bucheon City,Gyeonggi-do,421-150,48.236944,10,0,0,18,29,7,3,2022,10,2,0,20,29,7,3,2022,"2953,2953,2953",1,T1,2022-07-18 10:52:33+00:00,1,1,3,2953
1,1,dc670e53-0eb4-4da2-a023-8f505d74e961,2022-07-18 10:52:33+00:00,2022-07-20 11:05:38+00:00,10,3,0,4,product,Chrome,Email,Bucheon City,Gyeonggi-do,421-150,48.218056,10,0,0,18,29,7,3,2022,11,2,0,20,29,7,3,2022,"4731,4731,4731",2,T2,2022-07-18 11:18:36+00:00,1,1,3,4731
2,1,7ed34a21-9559-4d31-a16f-d87e4c22d343,2022-07-18 11:18:36+00:00,2022-07-19 11:29:28+00:00,10,3,0,4,department,Chrome,Email,Bucheon City,Gyeonggi-do,421-150,24.181111,11,0,0,18,29,7,3,2022,11,1,0,19,29,7,3,2022,"7656,7656,7656",3,T3,NaT,0,1,3,7656
3,100,5a795c3c-e0b6-40b3-a3ec-8dfc66157847,2023-12-09 13:20:07+00:00,2023-12-09 13:24:46+00:00,5,1,0,5,cart,Chrome,Email,Shanghai,Zhejiang,314300,0.077500,13,5,1,9,49,12,4,2023,13,5,1,9,49,12,4,2023,5742,1,T1,NaT,0,1,1,5742
4,1000,4faaee5f-3fa9-4665-9c36-cb073c42f189,2020-04-10 06:52:02+00:00,2020-04-10 06:59:30+00:00,5,1,0,5,department,Safari,YouTube,Seoul,Seoul,138-200,0.124444,6,4,0,10,15,4,2,2020,6,4,0,10,15,4,2,2020,1936,1,T1,NaT,0,1,1,1936


In [14]:
session_df.describe(include='all')

,user_id,session_id,session_start_time,session_end_time,session_event_count,session_product_views,session_cancel_events,event_type_nunique,session_main_events,session_browser,session_traffic_source,session_city,session_state,session_postal_code,session_duration_hour,session_start_hour,session_start_weekday,session_start_is_weekend,session_start_day,session_start_week,session_start_month,session_start_quarter,session_start_year,session_end_hour,session_end_weekday,session_end_is_weekend,session_end_day,session_end_week,session_end_month,session_end_quarter,session_end_year,product_seq_uri,session_order,session_type,next_session_start_time,label_repurchase,product_seq_unique_count,product_seq_total_count,product_seq_unique_str
count,177354,177354,177354,177354,177354.000000,177354.000000,177354.0,177354.000000,177354,177354,177354,175716,177354,177354,177354.000000,177354.000000,177354.000000,177354.000000,177354.000000,177354.0,177354.000000,177354.000000,177354.000000,177354.000000,177354.000000,177354.000000,177354.000000,177354.0,177354.000000,177354.000000,177354.000000,177354,177354.000000,177354,99217,177354.000000,177354.0,177354.000000,177354
unique,78137,177354,NaN,NaN,NaN,NaN,NaN,NaN,5,5,5,7555,228,14996,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,81286,NaN,14,NaN,NaN,NaN,NaN,29033
top,32996,36307bbf-664f-4ed4-9b98-85450cd5300e,NaN,NaN,NaN,NaN,NaN,NaN,cart,Chrome,Email,Shanghai,Guangdong,02675-031,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,21842,NaN,T1,NaN,NaN,NaN,NaN,18795
freq,14,1,NaN,NaN,NaN,NaN,NaN,NaN,48922,89172,79692,4496,9506,688,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,12,NaN,78137,NaN,NaN,NaN,NaN,21
mean,NaN,NaN,2022-10-12 13:19:02.244133120+00:00,2022-10-13 14:20:47.993971968+00:00,7.187833,1.901857,0.0,4.482261,NaN,NaN,NaN,NaN,NaN,NaN,25.029375,9.554552,3.010578,0.287803,15.518979,28.691825,7.026174,2.677149,2022.238613,9.524037,3.003665,0.286833,15.541849,28.591873,7.004618,2.670743,2022.243203,NaN,2.203198,NaN,2022-12-05 07:28:29.221111552+00:00,0.559429,1.0,1.901857,NaN
min,NaN,NaN,2019-01-06 02:16:07+00:00,2019-01-06 02:25:41+00:00,5.000000,1.000000,0.0,4.000000,NaN,NaN,NaN,NaN,NaN,NaN,0.004444,0.000000,0.000000,0.000000,1.000000,1.0,1.000000,1.000000,2019.000000,0.000000,0.000000,0.000000,1.000000,1.0,1.000000,1.000000,2019.000000,NaN,1.000000,NaN,2019-01-06 02:33:37+00:00,0.000000,1.0,1.000000,NaN
25%,NaN,NaN,2022-02-10 17:03:39.750000128+00:00,2022-02-12 00:06:39.249999872+00:00,5.000000,1.000000,0.0,4.000000,NaN,NaN,NaN,NaN,NaN,NaN,0.100000,4.000000,1.000000,0.000000,8.000000,15.0,4.000000,2.000000,2022.000000,4.000000,1.000000,0.000000,8.000000,15.0,4.000000,2.000000,2022.000000,NaN,1.000000,NaN,2022-05-14 03:30:02+00:00,0.000000,1.0,1.000000,NaN
50%,NaN,NaN,2023-02-06 08:35:14.500000+00:00,2023-02-07 09:58:50+00:00,7.000000,2.000000,0.0,4.000000,NaN,NaN,NaN,NaN,NaN,NaN,0.154167,9.000000,3.000000,0.000000,15.000000,30.0,7.000000,3.000000,2023.000000,9.000000,3.000000,0.000000,15.000000,30.0,7.000000,3.000000,2023.000000,NaN,2.000000,NaN,2023-04-01 12:04:03+00:00,1.000000,1.0,2.000000,NaN
75%,NaN,NaN,2023-09-13 05:24:35.500000+00:00,2023-09-14 05:23:56+00:00,7.000000,2.000000,0.0,5.000000,NaN,NaN,NaN,NaN,NaN,NaN,48.209167,14.000000,5.000000,1.000000,23.000000,43.0,10.000000,4.000000,2023.000000,14.000000,5.000000,1.000000,23.000000,43.0,10.000000,4.000000,2023.000000,NaN,3.000000,NaN,2023-10-06 09:42:54+00:00,1.000000,1.0,2.000000,NaN
max,NaN,NaN,2024-01-17 17:18:51+00:00,2024-01-21 16:18:30+00:00,13.000000,4.000000,0.0,5.000000,NaN,NaN,NaN,NaN,NaN,NaN,96.478611,23.000000,6.000000,1.000000,31.000000,53.0,12.000000,4.000000,2024.000000,23.000000,6.000000,1.000000,31.000000,53.0,12.000000,4.000000,2024.000000,NaN,14.000000,NaN,2024-01-17 17:18:51+00:00,1.000000,1.0,4.000000,NaN


In [15]:
session_df['session_main_events'].value_counts()    

session_main_events
cart          48922
department    47758
product       46909
home          17080
purchase      16685
Name: count, dtype: Int64

## 2.3 Master Table Joins & Feature Assembly

**products master join**

In [16]:
prod_info = (
    products[['id', 'retail_price', 'category']]
        .rename(columns={
            'id': 'product_seq_unique_str',
            'retail_price': 'product_retail_price',
            'category': 'product_category'
        })
)
prod_info['product_seq_unique_str'] = prod_info['product_seq_unique_str'].astype(str)
prod_info

,product_seq_unique_str,product_retail_price,product_category
0,13842,6.25,Accessories
1,13928,5.95,Accessories
2,14115,10.99,Accessories
3,14157,10.99,Accessories
4,14273,15.99,Accessories
...,...,...,...
29115,5676,24.17,Pants & Capris
29116,6538,25.00,Shorts
29117,6712,25.00,Shorts
29118,6821,25.00,Shorts


In [18]:
session_df = session_df.merge(
    prod_info,
    on='product_seq_unique_str',
    how='left'
)
session_df['total_price'] = session_df['product_retail_price'] * session_df['product_seq_total_count']
session_df.head()

,user_id,session_id,session_start_time,session_end_time,session_event_count,session_product_views,session_cancel_events,event_type_nunique,session_main_events,session_browser,session_traffic_source,session_city,session_state,session_postal_code,session_duration_hour,session_start_hour,session_start_weekday,session_start_is_weekend,session_start_day,session_start_week,session_start_month,session_start_quarter,session_start_year,session_end_hour,session_end_weekday,session_end_is_weekend,session_end_day,session_end_week,session_end_month,session_end_quarter,session_end_year,product_seq_uri,session_order,session_type,next_session_start_time,label_repurchase,product_seq_unique_count,product_seq_total_count,product_seq_unique_str,product_retail_price,product_category,total_price
0,1,bccf01cb-6f3b-4ef7-aaff-0ea67e584334,2022-07-18 10:17:52+00:00,2022-07-20 10:32:05+00:00,10,3,0,4,product,Firefox,Adwords,Bucheon City,Gyeonggi-do,421-150,48.236944,10,0,0,18,29,7,3,2022,10,2,0,20,29,7,3,2022,"2953,2953,2953",1,T1,2022-07-18 10:52:33+00:00,1,1,3,2953,15.00,Active,45.000000
1,1,dc670e53-0eb4-4da2-a023-8f505d74e961,2022-07-18 10:52:33+00:00,2022-07-20 11:05:38+00:00,10,3,0,4,product,Chrome,Email,Bucheon City,Gyeonggi-do,421-150,48.218056,10,0,0,18,29,7,3,2022,11,2,0,20,29,7,3,2022,"4731,4731,4731",2,T2,2022-07-18 11:18:36+00:00,1,1,3,4731,125.00,Jeans,375.000000
2,1,7ed34a21-9559-4d31-a16f-d87e4c22d343,2022-07-18 11:18:36+00:00,2022-07-19 11:29:28+00:00,10,3,0,4,department,Chrome,Email,Bucheon City,Gyeonggi-do,421-150,24.181111,11,0,0,18,29,7,3,2022,11,1,0,19,29,7,3,2022,"7656,7656,7656",3,T3,NaT,0,1,3,7656,19.99,Blazers & Jackets,59.969999
3,100,5a795c3c-e0b6-40b3-a3ec-8dfc66157847,2023-12-09 13:20:07+00:00,2023-12-09 13:24:46+00:00,5,1,0,5,cart,Chrome,Email,Shanghai,Zhejiang,314300,0.077500,13,5,1,9,49,12,4,2023,13,5,1,9,49,12,4,2023,5742,1,T1,NaT,0,1,1,5742,10.99,Leggings,10.990000
4,1000,4faaee5f-3fa9-4665-9c36-cb073c42f189,2020-04-10 06:52:02+00:00,2020-04-10 06:59:30+00:00,5,1,0,5,department,Safari,YouTube,Seoul,Seoul,138-200,0.124444,6,4,0,10,15,4,2,2020,6,4,0,10,15,4,2,2020,1936,1,T1,NaT,0,1,1,1936,15.00,Fashion Hoodies & Sweatshirts,15.000000


**User master join**

In [19]:
user_info = (
    users[['id', 'age', 'gender', 'state', 'street_address', 'postal_code', 'city', 'country', 'traffic_source']]
        .rename(columns={
            'id': 'user_id',
            'age': 'user_age',
            'gender': 'user_gender',
            'state': 'user_state',
            'street_address': 'user_street_address',
            'postal_code': 'user_postal_code',
            'city': 'user_city',
            'country': 'user_country',
            'traffic_source': 'user_traffic_source'
        })
)
user_info['user_id'] = user_info['user_id'].astype(str)
user_info

,user_id,user_age,user_gender,user_state,user_street_address,user_postal_code,user_city,user_country,user_traffic_source
0,457,65,M,Acre,87620 Johnson Hills,69917-400,Rio Branco,Brasil,Search
1,6578,34,F,Acre,1705 Nielsen Land,69917-400,Rio Branco,Brasil,Search
2,36280,13,M,Acre,125 Turner Isle Apt. 264,69917-400,Rio Branco,Brasil,Email
3,60193,64,M,Acre,0966 Jose Branch Apt. 008,69917-400,Rio Branco,Brasil,Search
4,64231,25,F,Acre,20798 Phillip Trail Apt. 392,69917-400,Rio Branco,Brasil,Search
...,...,...,...,...,...,...,...,...,...
99995,93247,36,F,Île-de-France,984 Brady Branch,77120,Beautheil-Saints,France,Search
99996,59110,12,M,Île-de-France,89560 Phillips Lakes Apt. 604,77160,Chenoise-Cucharmoy,France,Organic
99997,57045,53,F,Île-de-France,749 Ronald Forge,77320,Choisy-en-Brie,France,Search
99998,73312,16,F,Île-de-France,78117 Anderson Oval,77320,Choisy-en-Brie,France,Search


In [ ]:
session_df = session_df.merge(
    user_info,
    on='user_id',
    how='left'
)
session_df.head()

In [23]:
# 지역 및 트래픽 소스에 대해 세션 테이블과 유저 테이블이 일치하는지 확인 -- 둘 중 하나만 쓰고 파생 변수 생성 불가함
print(f"city: {(session_df['session_city'] == session_df['user_city']).value_counts()}")
print(f"state: {(session_df['session_state'] == session_df['user_state']).value_counts()}")
print(f"postal_code: {(session_df['session_postal_code'] == session_df['user_postal_code']).value_counts()}")
print(f"traffic_source: {(session_df['session_traffic_source'] == session_df['user_traffic_source']).value_counts()}")

city: True    175716
Name: count, dtype: Int64
state: True    177354
Name: count, dtype: Int64
postal_code: True    177354
Name: count, dtype: Int64
traffic_source: False    170911
True       6443
Name: count, dtype: Int64


## 2.4 Model input column setup

In [24]:
use_cols = [
    'user_id',
    'session_id',
    'session_event_count',
    'event_type_nunique',
    'session_main_events',
    'session_browser',
    'session_traffic_source',
    'session_state',
    'session_duration_hour',
    'session_start_hour',
    'session_start_weekday',
    'session_start_is_weekend',
    'session_start_day',
    'session_start_week',
    'session_start_month',
    'session_start_quarter',
    'session_start_year',
    'session_end_hour',
    'session_end_weekday',
    'session_end_is_weekend',
    'session_end_day',
    'session_end_week',
    'session_end_month',
    'session_end_quarter',
    'session_end_year',
    'product_seq_unique_str',
    'product_category',
    'total_price',
    'user_age',
    'user_gender',
    'user_country'
]

In [25]:
exclude_cols = [
    'session_start_time',
    'session_end_time',
    'session_product_views',
    'session_cancel_events',
    'session_city',
    'session_postal_code',
    'product_seq_uri',
    'product_seq_unique_count',
    'product_seq_total_count',
    'session_order',
    'session_type',
    'product_retail_price',
    'user_state',
    'user_street_address',
    'user_postal_code',
    'user_city',
    'user_traffic_source',
    'next_session_start_time'
]

In [26]:
target_col = [
    'label_repurchase'
]

In [27]:
print(len(use_cols))
print(len(exclude_cols))
print(len(target_col))
print(len(use_cols) + len(exclude_cols)+len(target_col))
print(session_df.shape)

31
18
1
50
(177354, 50)


**Session filtering**

In [28]:
# 시점 1 데이터에 대해서만 사용
session_df = session_df[session_df['session_type'] == 'T1']
session_df

,user_id,session_id,session_start_time,session_end_time,session_event_count,session_product_views,session_cancel_events,event_type_nunique,session_main_events,session_browser,session_traffic_source,session_city,session_state,session_postal_code,session_duration_hour,session_start_hour,session_start_weekday,session_start_is_weekend,session_start_day,session_start_week,session_start_month,session_start_quarter,session_start_year,session_end_hour,session_end_weekday,session_end_is_weekend,session_end_day,session_end_week,session_end_month,session_end_quarter,session_end_year,product_seq_uri,session_order,session_type,next_session_start_time,label_repurchase,product_seq_unique_count,product_seq_total_count,product_seq_unique_str,product_retail_price,product_category,total_price,user_age,user_gender,user_state,user_street_address,user_postal_code,user_city,user_country,user_traffic_source
0,1,bccf01cb-6f3b-4ef7-aaff-0ea67e584334,2022-07-18 10:17:52+00:00,2022-07-20 10:32:05+00:00,10,3,0,4,product,Firefox,Adwords,Bucheon City,Gyeonggi-do,421-150,48.236944,10,0,0,18,29,7,3,2022,10,2,0,20,29,7,3,2022,"2953,2953,2953",1,T1,2022-07-18 10:52:33+00:00,1,1,3,2953,15.000000,Active,45.000000,62,F,Gyeonggi-do,26092 John Well Suite 208,421-150,Bucheon City,South Korea,Search
3,100,5a795c3c-e0b6-40b3-a3ec-8dfc66157847,2023-12-09 13:20:07+00:00,2023-12-09 13:24:46+00:00,5,1,0,5,cart,Chrome,Email,Shanghai,Zhejiang,314300,0.077500,13,5,1,9,49,12,4,2023,13,5,1,9,49,12,4,2023,5742,1,T1,NaT,0,1,1,5742,10.990000,Leggings,10.990000,40,F,Zhejiang,3873 Miller Shore,314300,Shanghai,China,Search
4,1000,4faaee5f-3fa9-4665-9c36-cb073c42f189,2020-04-10 06:52:02+00:00,2020-04-10 06:59:30+00:00,5,1,0,5,department,Safari,YouTube,Seoul,Seoul,138-200,0.124444,6,4,0,10,15,4,2,2020,6,4,0,10,15,4,2,2020,1936,1,T1,NaT,0,1,1,1936,15.000000,Fashion Hoodies & Sweatshirts,15.000000,53,F,Seoul,31508 Natalie Junctions,138-200,Seoul,South Korea,Search
5,10000,eda1faa6-0c3a-4a42-a93c-cdb0b0211095,2022-01-06 10:16:47+00:00,2022-01-06 10:24:08+00:00,5,1,0,5,home,Chrome,Email,Connah's Quay,Wales,CH5,0.122500,10,3,0,6,1,1,1,2022,10,3,0,6,1,1,1,2022,2308,1,T1,NaT,0,1,1,2308,18.020000,Fashion Hoodies & Sweatshirts,18.020000,41,F,Wales,9324 Dawn Meadows Apt. 193,CH5,Connah's Quay,United Kingdom,Search
6,100000,b54dd218-3dd5-4a5d-bdcd-d8c6f06194d0,2020-08-28 23:21:02+00:00,2020-08-28 23:25:19+00:00,5,1,0,5,purchase,Chrome,Adwords,Beijing,Jiangsu,215337,0.071389,23,4,0,28,35,8,3,2020,23,4,0,28,35,8,3,2020,10017,1,T1,2021-08-21 00:40:06+00:00,1,1,1,10017,129.949997,Sleep & Lounge,129.949997,69,F,Jiangsu,637 French Forest Apt. 060,215337,Beijing,China,Display
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
177341,99992,6067c45a-de29-4038-8cf1-feb3d88155fa,2022-08-06 11:59:39+00:00,2022-08-08 12:09:54+00:00,10,3,0,4,cart,IE,Email,Clinton Township,Michigan,48035,48.170833,11,5,1,6,31,8,3,2022,12,0,0,8,32,8,3,2022,"11307,11307,11307",1,T1,2022-08-06 12:15:27+00:00,1,1,3,11307,12.000000,Intimates,36.000000,33,F,Michigan,3203 Brandon Shore,48035,Clinton Township,United States,Search
177344,99993,57b94289-c09a-4cba-bf9e-df3d4765b4bc,2024-01-03 02:12:19+00:00,2024-01-03 02:16:19+00:00,5,1,0,5,cart,Firefox,Adwords,Sydney,New South Wales,2133,0.066667,2,2,0,3,1,1,1,2024,2,2,0,3,1,1,1,2024,9826,1,T1,NaT,0,1,1,9826,26.000000,Sleep & Lounge,26.000000,17,F,New South Wales,42585 Powell Motorway,2133,Sydney,Australia,Search
177345,99996,7c8aa36e-a448-4bad-9ea8-d18a181d7359,2020-08-01 23:21:14+00:00,2020-08-01 23:33:20+00:00,7,2,0,4,product,Chrome,Email,Nanping,Guangdong,522021,0.201667,23,5,1,1,31,8,3,2020,23,5,1,1,31,8,3,2020,"27368,27368",1,T1,2020-08-02 00:05:56+00:00,1,1,2,27368,40.000000,Sleep & Lounge,80.000000,51,M,Guangdong,87464 Kathryn Islands Suite 457,522021,Nanping,China,Search
177349,99998,f900a0c8-62c5-4168-9ac8-cd0bde779778,2022-12-

**drop unnecessary fields**

In [29]:
session_df.drop(columns=exclude_cols, inplace=True)
session_df.head()

,user_id,session_id,session_event_count,event_type_nunique,session_main_events,session_browser,session_traffic_source,session_state,session_duration_hour,session_start_hour,session_start_weekday,session_start_is_weekend,session_start_day,session_start_week,session_start_month,session_start_quarter,session_start_year,session_end_hour,session_end_weekday,session_end_is_weekend,session_end_day,session_end_week,session_end_month,session_end_quarter,session_end_year,label_repurchase,product_seq_unique_str,product_category,total_price,user_age,user_gender,user_country
0,1,bccf01cb-6f3b-4ef7-aaff-0ea67e584334,10,4,product,Firefox,Adwords,Gyeonggi-do,48.236944,10,0,0,18,29,7,3,2022,10,2,0,20,29,7,3,2022,1,2953,Active,45.000000,62,F,South Korea
3,100,5a795c3c-e0b6-40b3-a3ec-8dfc66157847,5,5,cart,Chrome,Email,Zhejiang,0.077500,13,5,1,9,49,12,4,2023,13,5,1,9,49,12,4,2023,0,5742,Leggings,10.990000,40,F,China
4,1000,4faaee5f-3fa9-4665-9c36-cb073c42f189,5,5,department,Safari,YouTube,Seoul,0.124444,6,4,0,10,15,4,2,2020,6,4,0,10,15,4,2,2020,0,1936,Fashion Hoodies & Sweatshirts,15.000000,53,F,South Korea
5,10000,eda1faa6-0c3a-4a42-a93c-cdb0b0211095,5,5,home,Chrome,Email,Wales,0.122500,10,3,0,6,1,1,1,2022,10,3,0,6,1,1,1,2022,0,2308,Fashion Hoodies & Sweatshirts,18.020000,41,F,United Kingdom
6,100000,b54dd218-3dd5-4a5d-bdcd-d8c6f06194d0,5,5,purchase,Chrome,Adwords,Jiangsu,0.071389,23,4,0,28,35,8,3,2020,23,4,0,28,35,8,3,2020,1,10017,Sleep & Lounge,129.949997,69,F,China


# 3. Train/Test Split

## 3.1 Stratified split on label_repurchase

val set은 각자 분리

In [ ]:
# Stratified Split
train_df, test_df = train_test_split(
    session_df,
    test_size=0.2,              # 필요에 따라 변경
    stratify=session_df[target_col],   # 분포를 기준으로 층화 분할
    random_state=42
)

In [44]:
print("====All set====")
vc = session_df[target_col].value_counts()
out = pd.DataFrame({
    "count": vc.apply(lambda x: f"{x:,}"),
    "ratio": (vc / vc.sum()).round(3)
})
print(f"Total: {session_df.shape[0]:,}\n")
print(out)

print("====train set====")
vc = train_df[target_col].value_counts()
out = pd.DataFrame({
    "count": vc.apply(lambda x: f"{x:,}"),
    "ratio": (vc / vc.sum()).round(3)
})
print(f"Total: {train_df.shape[0]:,}\n")
print(out)

print("====test set====")
vc = test_df[target_col].value_counts()
out = pd.DataFrame({
    "count": vc.apply(lambda x: f"{x:,}"),
    "ratio": (vc / vc.sum()).round(3)
})

print(f"Total: {test_df.shape[0]:,}\n")
print(out)

====All set====
Total: 78,137

                   count  ratio
label_repurchase               
1                 43,953  0.563
0                 34,184  0.437
====train set====
Total: 62,509

                   count  ratio
label_repurchase               
1                 35,162  0.563
0                 27,347  0.437
====test set====
Total: 15,628

                  count  ratio
label_repurchase              
1                 8,791  0.563
0                 6,837  0.437


In [ ]:
prc_path

## 3.2 Save train/test tables

In [45]:
# 파일명 정의
train_csv_path = f"{prc_path}session_train.csv"
test_csv_path  = f"{prc_path}session_test.csv"

# 저장 (CSV)
train_df.to_csv(train_csv_path, index=False, encoding="utf-8")
test_df.to_csv(test_csv_path, index=False, encoding="utf-8")

print("Train/Test tables saved:")
print(f"- {train_csv_path}")
print(f"- {test_csv_path}")

Train/Test tables saved:
- C:/Users/Owner/Desktop/지윤/3.기타/프로젝트/purchase-conversion-prediction/data/processed/session_train.csv
- C:/Users/Owner/Desktop/지윤/3.기타/프로젝트/purchase-conversion-prediction/data/processed/session_test.csv


# 4. EDA